# 02 — Public-data calibration and synthetic auctions

**Specification:** week 6 Introduction and Methods. Public Larsen data only; no employer data. Original assets are read-only. This notebook fits distributional models and generates 50,000 new vehicle records per scenario. It is a calibration stage, not a fitted-policy evaluation.

The primary scenario has a binding floor. A bargaining comparison permits accepted transactions below reserve. The institutional difference is measured explicitly: matching all bargaining price distributions with a binding floor may be impossible. Estimated observational associations are not identified causal intervention responses. Private-information strength, the declining floor effect, market shocks, and continuation costs remain stated assumptions.

Run all cells from `code-repo`. Exports go to `data/calibration/` and `data/synthetic/`. Rerunning replaces these generated files. Source run identifiers, seller identifiers, and VIN fragments are never exported. Historical dollars and source calendar coverage are retained; there is no inflation adjustment.


In [1]:
from pathlib import Path
import hashlib, json, platform, importlib.metadata
import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.optimize import brentq
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge, LogisticRegression
from IPython.display import display
SOURCE = Path('Larsen_used_car_bargaining_data_and_code')
CAL = Path('data/calibration'); SYN = Path('data/synthetic')
CAL.mkdir(parents=True, exist_ok=True); SYN.mkdir(parents=True, exist_ok=True)
CONFIG = dict(n=50000, seed=6940, floor_effect=-4.0, private_floor_strength=0.08,
              private_sale_strength=0.30, private_price_strength=0.04,
              market_sd=0.12, holding_cost=300.0, retained_ratio=0.85,
              ridge_alpha=10.0, logit_C=1.0)
rng = np.random.default_rng(CONFIG['seed'])
FILES = {'dealer':'pre_step1_ins0.csv', 'fleet_lease':'pre_step1_ins1.csv'}
def sha(path):
    with path.open('rb') as f: return hashlib.file_digest(f, 'sha256').hexdigest()
source_hashes = {p.name:sha(p) for p in SOURCE.iterdir() if p.is_file()}
display(pd.Series(CONFIG, name='settings'))


n                         50000.00
seed                       6940.00
floor_effect                 -4.00
private_floor_strength        0.08
private_sale_strength         0.30
private_price_strength        0.04
market_sd                     0.12
holding_cost                300.00
retained_ratio                0.85
ridge_alpha                  10.00
logit_C                       1.00
Name: settings, dtype: float64

## 1. Prepare eligible calibration populations

Eligibility is purpose-specific and counted sequentially. Base characteristics require known date, nonnegative age and mileage, and a positive benchmark. Missing condition/make/location receive an explicit category. Outcome models additionally require a known outcome and a positive reserve. Zero reserves are excluded from the **log-floor** fit and counted. Unknown outcomes are not imputed.

The synthetic seller-group mix follows eligible source run counts, an explicit sampling assumption; it is not an estimate of the unique-vehicle population mix. `idnew` cannot establish longitudinal histories. We use make as an observed vehicle category; a true segment classification is unavailable and is not invented.


In [2]:
sources, exclusions = {}, []
for group, filename in FILES.items():
    cols = ['history_run_date','year','odo','bb','make','conrepgrade','auction',
            'low','agree','finalprice','salefee','offeramt1']
    d = pd.read_csv(SOURCE/filename, usecols=cols, low_memory=False)
    d['date'] = pd.to_datetime(d.history_run_date, format='%d%b%Y', errors='coerce')
    d['age'] = d.date.dt.year - d.year
    d = d.rename(columns={'odo':'mileage','bb':'benchmark','conrepgrade':'condition','auction':'location'})
    mask = pd.Series(True, index=d.index)
    rules = {'valid_date':d.date.notna(), 'valid_age':d.age.ge(0) & np.isfinite(d.age),
             'valid_mileage':d.mileage.ge(0) & np.isfinite(d.mileage),
             'positive_benchmark':d.benchmark.gt(0) & np.isfinite(d.benchmark)}
    for rule, ok in rules.items():
        exclusions.append(dict(group=group, stage=rule, excluded=int((mask & ~ok).sum())))
        mask &= ok
    d = d.loc[mask].copy()
    for name in ['make','condition','location']: d[name] = d[name].fillna('<missing>').astype(str)
    d['month'] = d.date.dt.to_period('M').astype(str)
    d['floor_ratio'] = d.low / d.benchmark
    d['price_ratio'] = d.finalprice.where(d.agree.eq(1) & d.finalprice.gt(0)) / d.benchmark
    d['seller_group'] = group
    d['outcome_eligible'] = d.floor_ratio.gt(0) & np.isfinite(d.floor_ratio) & d.agree.isin([0,1])
    exclusions.append(dict(group=group, stage='outcome_floor_eligibility_after_base', excluded=int((~d.outcome_eligible).sum())))
    sources[group] = d
pd.DataFrame(exclusions).to_csv(CAL/'exclusions.csv',index=False)
display(pd.DataFrame(exclusions))


,group,stage,excluded
0,dealer,valid_date,0
1,dealer,valid_age,0
2,dealer,valid_mileage,0
3,dealer,positive_benchmark,0
4,dealer,outcome_floor_eligibility_after_base,19142
5,fleet_lease,valid_date,0
6,fleet_lease,valid_age,0
7,fleet_lease,valid_mileage,0
8,fleet_lease,positive_benchmark,0
9,fleet_lease,outcome_floor_eligibility_after_base,65179


## 2. Fit distributional models and record parameters

Generate age, make, condition, location, and calendar month jointly from aggregated source frequency tables. Generate mileage conditionally on age and make; generate benchmark conditionally on vehicle characteristics and month. Log-scale residuals are sampled within seller group, with 0.5%/99.5% residual tail truncation and source 0.1%/99.9% output bounds. These stabilization choices are assumptions and affect the tails.

Historical floor assignment uses a conditional log-ratio regression plus residual variation and private information. The randomized comparison permutes those floors within seller group, breaking their connection with the current vehicle and private signal.

The conversion model estimates observable associations from eligible source runs. Its fitted floor coefficient is exported for inspection but **replaced** in the generator by the prespecified decreasing effect. The intercept is calibrated to the eligible source conversion rate. Bargaining prices use all eligible positive sales. Binding-floor premiums use only positive above-floor price differences, a selected subset—not a fit to all source transactions. Fee assumptions use group medians on eligible sales and are known before the simulated auction.


In [3]:
def model_frame(d):
    x = d.copy()
    x['log_mileage'] = np.log1p(x.mileage)
    x['log_benchmark'] = np.log(x.benchmark)
    return x

def estimator(numeric, categorical, logistic=False):
    prep = ColumnTransformer([
        ('num', StandardScaler(), numeric),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)])
    fit = LogisticRegression(C=CONFIG['logit_C'], max_iter=500) if logistic else Ridge(alpha=CONFIG['ridge_alpha'], solver='lsqr', tol=1e-6)
    return make_pipeline(prep, fit)

def fit_log(d, y, numeric, categorical):
    m = estimator(numeric, categorical)
    m.fit(d, y)
    residual = np.asarray(y)-m.predict(d)
    lo,hi = np.quantile(residual,[.005,.995])
    return m, np.clip(residual,lo,hi)

def parameter_rows(m, group, name):
    names=m.steps[0][1].get_feature_names_out()
    fit=m.steps[-1][1]
    rows=[dict(group=group,model=name,term='intercept',coefficient=float(np.ravel(fit.intercept_)[0]))]
    rows += [dict(group=group,model=name,term=str(k),coefficient=float(v)) for k,v in zip(names,np.ravel(fit.coef_))]
    return rows

fits, coefficient_rows, target_rows, frequency_tables = {}, [], [], []
for group, raw in sources.items():
    d=model_frame(raw); e=d.loc[d.outcome_eligible].copy()
    cats=['make','condition','location','month']; nums=['age','log_mileage','log_benchmark']
    joint=d.groupby(['age','make','condition','location','month'],dropna=False).size().reset_index(name='count')
    frequency_tables.append(joint.assign(seller_group=group))
    mileage=fit_log(d,np.log1p(d.mileage),['age'],['make'])
    benchmark=fit_log(d,np.log(d.benchmark),['age','log_mileage'],cats)
    floor=fit_log(e,np.log(e.floor_ratio),nums,cats)
    conversion=estimator(nums+['floor_ratio'],cats,logistic=True)
    conversion.fit(e,e.agree)
    sold=e.loc[e.agree.eq(1) & e.price_ratio.gt(0)].copy()
    price=fit_log(sold,np.log(sold.price_ratio),nums+['floor_ratio'],cats)
    premium=sold.price_ratio-sold.floor_ratio
    above=sold.loc[premium.gt(0)].copy()
    binding=fit_log(above,np.log(premium[premium.gt(0)]),nums+['floor_ratio'],cats)
    fee=float(sold.loc[sold.salefee.ge(0),'salefee'].median())
    fits[group]=dict(joint=joint,mileage=mileage,benchmark=benchmark,floor=floor,
                     conversion=conversion,price=price,binding=binding,fee=fee,
                     conversion_target=float(e.agree.mean()), source=e)
    for name in ['mileage','benchmark','floor','price','binding']:
        coefficient_rows += parameter_rows(fits[group][name][0],group,name)
    coefficient_rows += parameter_rows(conversion,group,'conversion_observational')
    for field in ['mileage','benchmark','floor_ratio']:
        values=d[field] if field!='floor_ratio' else e.floor_ratio
        fits[group][field+'_bounds']=np.quantile(values,[.001,.999]).tolist()
    target_rows.append(dict(group=group,base_n=len(d),outcome_n=len(e),sales_n=len(sold),
                            positive_premium_n=len(above),conversion=e.agree.mean(),
                            price_ratio_mean=sold.price_ratio.mean(),known_sale_fee=fee))
coefficients=pd.DataFrame(coefficient_rows)
coefficients.to_csv(CAL/'model_coefficients.csv',index=False)
pd.concat(frequency_tables).to_csv(CAL/'joint_category_frequencies.csv',index=False)
targets=pd.DataFrame(target_rows); targets.to_csv(CAL/'calibration_targets.csv',index=False)
display(targets)


,group,base_n,outcome_n,sales_n,positive_premium_n,conversion,price_ratio_mean,known_sale_fee
0,dealer,300740,281598,94170,10387,0.334413,0.980378,140.0
1,fleet_lease,211656,146477,100983,39840,0.689412,0.941589,95.0


## 3. Generate observable vehicles and isolated latent information

New identifiers represent one opportunity per synthetic vehicle. Dates are sampled within the selected historical month and source date coverage. There is no longitudinal vehicle reconstruction. Group-level fee schedules, holding costs, and retained values are decision-time inputs. Latent private signals and common daily market shocks are kept out of the observable CSVs.

These synthetic covariates retain source-supported categorical combinations; continuous quantities are newly drawn. Random residuals and assumptions are recorded for reproducibility.


In [4]:
group_sizes=rng.multinomial(CONFIG['n'],np.array([len(sources[g]) for g in FILES])/sum(map(len,sources.values())))
frames=[]; latent_frames=[]; generation_diagnostics=[]
for (group,f), n in zip(fits.items(),group_sizes):
    joint=f['joint']; idx=rng.choice(len(joint),n,p=joint['count']/joint['count'].sum())
    x=joint.iloc[idx].drop(columns='count').reset_index(drop=True).copy()
    x['seller_group']=group
    x['mileage']=0.0; x['benchmark']=1.0
    for field,transform in [('mileage',np.expm1),('benchmark',np.exp)]:
        m,residual=f[field]
        raw=transform(m.predict(model_frame(x))+rng.choice(residual,n))
        low,high=f[field+'_bounds']
        x[field]=np.clip(raw,max(0,low),high)
        generation_diagnostics.append(dict(group=group,field=field,clipped=int(((raw<max(0,low))|(raw>high)).sum()),n=int(n)))
    private=rng.normal(size=n)
    m,residual=f['floor']
    floor_raw=np.exp(m.predict(model_frame(x))+rng.choice(residual,n)+CONFIG['private_floor_strength']*private)
    x['floor_ratio']=np.clip(floor_raw,*f['floor_ratio_bounds'])
    generation_diagnostics.append(dict(group=group,field='floor_ratio',clipped=int((x.floor_ratio.to_numpy()!=floor_raw).sum()),n=int(n)))
    start=pd.to_datetime(x.month+'-01')
    days=start.dt.days_in_month.to_numpy()
    dates=start+pd.to_timedelta((rng.random(n)*days).astype(int),unit='D')
    x['auction_date']=dates.clip(lower=sources[group].date.min(),upper=sources[group].date.max())
    x['known_sale_fee']=f['fee'];x['holding_cost']=CONFIG['holding_cost'];x['retained_value']=CONFIG['retained_ratio']*x.benchmark
    frames.append(x)
    latent_frames.append(pd.DataFrame({'private_signal':private}))
vehicles=pd.concat(frames,ignore_index=True)
latents=pd.concat(latent_frames,ignore_index=True)
vehicles['vehicle_id']=[f'SYN-{i:06}' for i in range(len(vehicles))]
vehicles['run_id']=vehicles.vehicle_id+'-1';vehicles['run_number']=1
latents['vehicle_id']=vehicles.vehicle_id
unique_dates=np.sort(vehicles.auction_date.unique())
shocks=dict(zip(unique_dates,rng.normal(0,CONFIG['market_sd'],len(unique_dates))))
latents['market_shock']=vehicles.auction_date.map(shocks).to_numpy()
latents['sale_uniform']=rng.random(len(vehicles))
latents['price_residual_quantile']=rng.random(len(vehicles))
latents['bid_uniform']=rng.random(len(vehicles))
pd.DataFrame(generation_diagnostics).to_csv(CAL/'generation_clipping.csv',index=False)
assert len(vehicles)==50000
assert vehicles.vehicle_id.is_unique
display(vehicles.head())


,age,make,condition,location,month,seller_group,mileage,benchmark,floor_ratio,auction_date,known_sale_fee,holding_cost,retained_value,vehicle_id,run_id,run_number
0,6,CHEV,No grade,4,2007-02,dealer,84399.943013,7766.679533,1.211610,2007-02-10,140.0,300.0,6601.677603,SYN-000000,SYN-000000-1,1
1,11,FORD,No grade,1,2007-09,dealer,121908.753125,3746.732302,0.592144,2007-09-08,140.0,300.0,3184.722457,SYN-000001,SYN-000001-1,1
2,5,GMC,No grade,3,2008-08,dealer,54038.716270,7953.516295,1.084333,2008-08-07,140.0,300.0,6760.488851,SYN-000002,SYN-000002-1,1
3,9,DODG,No grade,3,2009-03,dealer,152944.755763,3345.964105,1.213452,2009-03-17,140.0,300.0,2844.069490,SYN-000003,SYN-000003-1,1
4,5,FORD,No grade,1,2008-06,dealer,147821.454243,6077.459673,1.590334,2008-06-18,140.0,300.0,5165.840722,SYN-000004,SYN-000004-1,1


## 4. Sale-response calibration and outcome scenarios

Calibrate a seller-group intercept to match eligible historical conversion in expectation for generated historical assignments, integrating over the actual simulated private signals and market effects. This is a distributional calibration, not estimation of the causal reserve effect. Keep that intercept fixed in the randomized-floor comparison.

Binding prices follow `floor + benchmark × exp(log-premium)`. A positive multiplicative price-component adjustment matches each group’s source mean price ratio under expected sale weighting when feasible; if the binding floor alone exceeds the target, the target is flagged unattainable and the fitted premium is retained. This mean calibration does not guarantee matching quantiles or below-reserve frequencies. Bargaining prices follow the fitted log price-ratio distribution. Private-price effects enter both scenarios. Sale prices are therefore positive. Generated initial offers obey the chosen trading rule; their allocation below reserve in bargaining is an explicit assumption, not a reproduction of Larsen's negotiation sequence.

The latent file contains conditional true probabilities and conditional price means. An eventual observable-information oracle must integrate **jointly** over private information and market uncertainty, not multiply separately averaged sale and price functions. The latent values must not enter predictive model features.


In [5]:
intercepts={}; price_calibration={}; scenario_frames={}; evaluation_frames=[]
for assignment in ['historical','randomized']:
    x=vehicles.copy()
    if assignment=='randomized':
        for group in FILES:
            ix=x.index[x.seller_group.eq(group)]
            x.loc[ix,'floor_ratio']=rng.permutation(x.loc[ix,'floor_ratio'].to_numpy())
    q=np.empty(len(x));means={};prices={}
    for group,f in fits.items():
        ix=x.index[x.seller_group.eq(group)]; z=model_frame(x.loc[ix])
        conv=f['conversion']; prep=conv.steps[0][1]; fit=conv.steps[-1][1]
        scale=prep.named_transformers_['num'].scale_[-1]
        floor_coefficient=fit.coef_[0,len(['age','log_mileage','log_benchmark'])]/scale
        base=conv.decision_function(z)-floor_coefficient*z.floor_ratio.to_numpy()+CONFIG['floor_effect']*z.floor_ratio.to_numpy()
        base+=CONFIG['private_sale_strength']*latents.loc[ix,'private_signal'].to_numpy()+latents.loc[ix,'market_shock'].to_numpy()
        if assignment=='historical':
            intercepts[group]=float(brentq(lambda b:expit(base+b).mean()-f['conversion_target'],-100,100))
        q[ix]=expit(base+intercepts[group])
    for institution in ['binding','bargaining']:
        price=np.empty(len(x));mean=np.empty(len(x))
        for group,f in fits.items():
            ix=x.index[x.seller_group.eq(group)];z=model_frame(x.loc[ix])
            m,residual=f['binding' if institution=='binding' else 'price']
            location=m.predict(z)+CONFIG['private_price_strength']*latents.loc[ix,'private_signal'].to_numpy()
            epsilon=np.quantile(residual,latents.loc[ix,'price_residual_quantile'].to_numpy())
            # Same interpolated empirical residual distribution for draws and analytic means.
            ordered=np.sort(residual); delta=np.diff(ordered)
            interval_mean=np.exp(ordered[:-1])*np.divide(np.expm1(delta),delta,out=np.ones_like(delta),where=delta!=0)
            smear=float(interval_mean.mean())
            offset=z.floor_ratio.to_numpy() if institution=='binding' else 0.0
            key=group+'_'+institution
            if assignment=='historical':
                target=float(f['source'].loc[f['source'].agree.eq(1), 'price_ratio'].mean())
                weighted_offset=float(np.average(np.broadcast_to(offset,len(ix)),weights=q[ix]))
                weighted_component=float(np.average(np.exp(location)*smear,weights=q[ix]))
                multiplier=(target-weighted_offset)/weighted_component
                feasible=multiplier>0
                price_calibration[key]={'target_mean_price_ratio':target,'minimum_mean_from_floor':weighted_offset,
                                        'target_attainable':bool(feasible),'multiplier':float(multiplier if feasible else 1.0)}
            location=location+np.log(price_calibration[key]['multiplier'])
            price[ix]=z.benchmark.to_numpy()*(offset+np.exp(location+epsilon))
            mean[ix]=z.benchmark.to_numpy()*(offset+np.exp(location)*smear)
        sold=latents.sale_uniform.to_numpy()<q
        result=x.copy();result['floor']=result.floor_ratio*result.benchmark
        result['sold']=sold.astype(int);result['price']=np.where(sold,price,np.nan)
        result['bargaining_accepted_below_floor']=sold & (price<result.floor)
        result['high_bid']=np.where(sold,np.minimum(price,result.floor) if institution=='bargaining' else price,
                                   result.floor*latents.bid_uniform)
        result['scenario']=institution+'_'+assignment
        result=result.drop(columns='month')
        name=institution+'_'+assignment
        scenario_frames[name]=result
        ev=latents.copy();ev['scenario']=name;ev['true_sale_probability_given_latents']=q
        ev['true_conditional_price_mean_given_latents']=mean
        evaluation_frames.append(ev)
        result.to_csv(SYN/(name+'.csv'),index=False)
evaluation=pd.concat(evaluation_frames,ignore_index=True)
evaluation.to_csv(SYN/'evaluation_only.csv',index=False)


## 5. Source-versus-synthetic diagnostics

Compare each seller group separately. Covariates target the base-characteristic sample; floor and outcome targets use the eligible reserve/outcome sample. Different denominators are deliberate and exported. Quantile differences, category total-variation distances, and correlation differences are diagnostics, not claims of a perfect fit. The binding price comparison uses **all eligible source sales** so institutional mismatch remains visible. Calibration to aggregate conversion does not validate subgroup conversion or causal responses.


In [6]:
comparison=[]; distributions=[]; category_comparison=[]; correlation_comparison=[]
for name,s in scenario_frames.items():
    for group,f in fits.items():
        g=s[s.seller_group.eq(group)]; src=f['source']; sold=g.sold.eq(1)
        source_sales=src.loc[src.agree.eq(1)&src.price_ratio.gt(0)]
        comparison.append(dict(scenario=name,group=group,n=len(g),source_conversion=f['conversion_target'],
          expected_conversion=evaluation.loc[evaluation.scenario.eq(name)&evaluation.vehicle_id.isin(g.vehicle_id),'true_sale_probability_given_latents'].mean(),
          realized_conversion=g.sold.mean(),source_mean_price_ratio=source_sales.price_ratio.mean(),
          synthetic_mean_price_ratio=(g.loc[sold,'price']/g.loc[sold,'benchmark']).mean(),
          source_below_reserve_share=(source_sales.finalprice<source_sales.low).mean(),
          synthetic_below_reserve_share=g.loc[sold,'bargaining_accepted_below_floor'].mean()))
        for field in ['age','mileage','benchmark','floor_ratio','price_ratio']:
            a=(source_sales.price_ratio if field=='price_ratio' else (src[field] if field=='floor_ratio' else sources[group][field])).dropna()
            b=(g.loc[sold,'price']/g.loc[sold,'benchmark'] if field=='price_ratio' else g[field]).dropna()
            for quantile in [.01,.05,.25,.5,.75,.95,.99]:
                av,bv=a.quantile(quantile),b.quantile(quantile)
                distributions.append(dict(scenario=name,group=group,field=field,quantile=quantile,source=av,synthetic=bv,difference=bv-av))
        for field in ['make','condition','location']:
            a=sources[group][field].value_counts(normalize=True);b=g[field].value_counts(normalize=True)
            ix=a.index.union(b.index)
            category_comparison.append(dict(scenario=name,group=group,field=field,total_variation=float((a.reindex(ix,fill_value=0)-b.reindex(ix,fill_value=0)).abs().sum()/2)))
        for first,second in [('age','mileage'),('age','benchmark'),('mileage','benchmark'),('benchmark','floor_ratio')]:
            a=sources[group][[first,second]].replace([np.inf,-np.inf],np.nan).dropna()
            correlation_comparison.append(dict(scenario=name,group=group,first=first,second=second,source=a[first].corr(a[second]),synthetic=g[first].corr(g[second])))
summary=pd.DataFrame(comparison)
summary.to_csv(CAL/'scenario_comparison.csv',index=False)
pd.DataFrame(distributions).to_csv(CAL/'quantile_comparison.csv',index=False)
pd.DataFrame(category_comparison).to_csv(CAL/'category_comparison.csv',index=False)
pd.DataFrame(correlation_comparison).to_csv(CAL/'correlation_comparison.csv',index=False)
display(summary)
display(pd.DataFrame(correlation_comparison).query("scenario == 'binding_historical'"))


,scenario,group,n,source_conversion,expected_conversion,realized_conversion,source_mean_price_ratio,synthetic_mean_price_ratio,source_below_reserve_share,synthetic_below_reserve_share
0,binding_historical,dealer,29340,0.334413,0.334413,0.334969,0.980378,0.985600,0.865116,0.000000
1,binding_historical,fleet_lease,20660,0.689412,0.689412,0.693320,0.941589,0.943486,0.559193,0.000000
2,bargaining_historical,dealer,29340,0.334413,0.334413,0.334969,0.980378,0.984946,0.865116,0.456451
3,bargaining_historical,fleet_lease,20660,0.689412,0.689412,0.693320,0.941589,0.945176,0.559193,0.463139
4,binding_randomized,dealer,29340,0.334413,0.349148,0.350170,0.980378,0.968646,0.865116,0.000000
5,binding_randomized,fleet_lease,20660,0.689412,0.681157,0.686012,0.941589,0.932130,0.559193,0.000000
6,bargaining_randomized,dealer,29340,0.334413,0.349148,0.350170,0.980378,0.973405,0.865116,0.451333
7,bargaining_randomized,fleet_lease,20660,0.689412,0.681157,0.686012,0.941589,0.940877,0.559193,0.440838


,scenario,group,first,second,source,synthetic
0,binding_historical,dealer,age,mileage,0.712585,0.732898
1,binding_historical,dealer,age,benchmark,-0.682169,-0.660438
2,binding_historical,dealer,mileage,benchmark,-0.542146,-0.490280
3,binding_historical,dealer,benchmark,floor_ratio,-0.304425,-0.344361
4,binding_historical,fleet_lease,age,mileage,0.801863,0.784268
5,binding_historical,fleet_lease,age,benchmark,-0.654330,-0.630124
6,binding_historical,fleet_lease,mileage,benchmark,-0.628935,-0.518582
7,binding_historical,fleet_lease,benchmark,floor_ratio,-0.215377,-0.195491


## 6. Provenance, reproducibility, and checks

Fitted distributional models are saved with the source fingerprints and feature preprocessing; they are for trusted local reuse only. No source identifiers are retained. Source-calibrated associations, calibration constraints, and structural assumptions are distinct. This notebook does not implement downstream predictive training, optimization, 30-replication evaluation, or the repeated-auction extension.


In [7]:
import joblib
provenance=pd.DataFrame([
 ('joint categorical frequencies','estimated','Age/make/condition/location/month frequencies by seller group'),
 ('seller-group mixture','assumption','Eligible source run proportions used for synthetic unique-vehicle mix'),
 ('make as vehicle category','assumption','Actual segment is unavailable; make is not renamed segment'),
 ('mileage and benchmark','estimated','Conditional log regressions and empirical residuals by group'),
 ('historical floor distribution','estimated + assumption','Conditional log regression plus assumed private-signal loading'),
 ('floor causal effect','assumption',str(CONFIG['floor_effect'])),
 ('observable sale associations','estimated','Penalized logistic coefficients; not causal identification'),
 ('conversion intercept','calibrated','Matches eligible source conversion on generated historical assignments'),
 ('bargaining price','estimated + assumption','Log price-ratio fit; assumed private loading and counterfactual interpretation'),
 ('price mean adjustment','calibrated','Group-specific expected price-ratio mean matched after sale selection when mathematically attainable'),
 ('binding premium','estimated on selected subset','Positive price-minus-floor subset; cannot reproduce all bargaining sales'),
 ('fee schedule','estimated + assumption','Group median realized sale fee used as known fixed fee'),
 ('tail bounds','assumption','Residual .005/.995 and generated .001/.999 source bounds'),
 ('private signals and daily shocks','assumption',json.dumps({k:v for k,v in CONFIG.items() if 'private' in k or 'market' in k})),
 ('holding and retained value','assumption','300 dollars holding cost; retained value 0.85 times benchmark'),
 ('initial high bid','assumption','Consistent with scenario outcomes; not calibrated to offeramt1 distribution'),
],columns=['parameter','provenance','details'])
provenance.to_csv(CAL/'parameter_provenance.csv',index=False)
# Save estimators plus residuals and bounds, excluding source records and identifiers.
saved={g:{k:v for k,v in f.items() if k!='source'} for g,f in fits.items()}
joblib.dump(saved,CAL/'fitted_calibration.joblib')
metadata={'config':CONFIG,'conversion_intercept_adjustments':intercepts,'price_calibration':price_calibration,'source_sha256':source_hashes,
          'packages':{p:importlib.metadata.version(p) for p in ['numpy','pandas','scipy','scikit-learn','joblib']},
          'python':platform.python_version(),'source_files':FILES,'scope':'single-run source-calibrated development scenarios; diagnostics required'}
(CAL/'calibration_metadata.json').write_text(json.dumps(metadata,indent=2)+'\n')
for name,s in scenario_frames.items():
    assert len(s)==CONFIG['n'] and s.vehicle_id.is_unique
    assert s.benchmark.gt(0).all() and s.mileage.ge(0).all()
    assert s.loc[s.sold.eq(0),'price'].isna().all()
    assert s.loc[s.sold.eq(1),'price'].gt(0).all()
    assert s.loc[s.sold.eq(0),'high_bid'].lt(s.loc[s.sold.eq(0),'floor']).all()
    if name.startswith('binding'):
        assert (s.loc[s.sold.eq(1),'price']>=s.loc[s.sold.eq(1),'floor']).all()
    assert not {'private_signal','market_shock','sale_uniform'} & set(s.columns)
assert evaluation.true_sale_probability_given_latents.between(0,1).all()
assert all(sha(SOURCE/name)==digest for name,digest in source_hashes.items())
assert (summary.loc[summary.scenario.str.endswith('historical'),'expected_conversion']-summary.loc[summary.scenario.str.endswith('historical'),'source_conversion']).abs().max()<1e-8
print('Checks passed: 50,000 records per scenario, outcome consistency, latent separation, conversion calibration, source hashes unchanged.')
display(provenance)


Checks passed: 50,000 records per scenario, outcome consistency, latent separation, conversion calibration, source hashes unchanged.


,parameter,provenance,details
0,joint categorical frequencies,estimated,Age/make/condition/location/month frequencies ...
1,seller-group mixture,assumption,Eligible source run proportions used for synth...
2,make as vehicle category,assumption,Actual segment is unavailable; make is not ren...
3,mileage and benchmark,estimated,Conditional log regressions and empirical resi...
4,historical floor distribution,estimated + assumption,Conditional log regression plus assumed privat...
5,floor causal effect,assumption,-4.0
6,observable sale associations,estimated,Penalized logistic coefficients; not causal id...
7,conversion intercept,calibrated,Matches eligible source conversion on generate...
8,bargaining price,estimated + assumption,Log price-ratio fit; assumed private loading a...
9,price mean adjustment,calibrated,Group-specific expected price-ratio mean match...


In [8]:
dictionary = pd.DataFrame([
 ('vehicle_id','generated identifier','New synthetic vehicle; not source idnew'),
 ('run_id','generated identifier','One synthetic auction opportunity'),
 ('run_number','integer','Always 1 in this stage'),
 ('auction_date','historical calendar date','Generated within sampled month and source coverage'),
 ('age','years','Joint categorical-frequency draw'),
 ('make','source categorical code','Observed make category; not a segment mapping'),
 ('condition','source categorical label','Includes No grade'),
 ('location','source auction code','Not a geographic name'),
 ('seller_group','dealer or fleet_lease','Mixture based on eligible source run counts'),
 ('mileage','miles','Conditional log regression and residual draw'),
 ('benchmark','historical dollars','Conditional log regression and residual draw'),
 ('floor_ratio','ratio','Historical conditional or randomized assignment'),
 ('floor','historical dollars','floor_ratio times benchmark'),
 ('known_sale_fee','historical dollars','Assumed known schedule from source sale-fee median'),
 ('holding_cost','historical dollars','Scenario assumption'),
 ('retained_value','historical dollars','Before additional holding cost'),
 ('sold','0 or 1','Bernoulli draw under specified response surface'),
 ('price','historical dollars','Missing for no-sale'),
 ('high_bid','historical dollars','Rule-consistent assumption; not source-calibrated'),
 ('bargaining_accepted_below_floor','boolean','True only for completed below-floor transactions'),
 ('scenario','label','Trading institution and floor assignment'),
],columns=['field','unit','definition'])
dictionary.to_csv(CAL/'synthetic_data_dictionary.csv',index=False)
for key, info in price_calibration.items():
    print(key, info)
# Re-read exported artifacts to verify schema and JSON round-trip.
json.loads((CAL/'calibration_metadata.json').read_text())
for name in scenario_frames:
    stored=pd.read_csv(SYN/(name+'.csv'))
    assert len(stored)==CONFIG['n'] and set(stored.columns)==set(dictionary.field)
print('Export schemas and metadata verified.')


dealer_binding {'target_mean_price_ratio': 0.980378420107658, 'minimum_mean_from_floor': 0.959046311843033, 'target_attainable': True, 'multiplier': 0.19826140695198458}
fleet_lease_binding {'target_mean_price_ratio': 0.9415885956004653, 'minimum_mean_from_floor': 0.9251162741100999, 'target_attainable': True, 'multiplier': 0.12780307658251988}
dealer_bargaining {'target_mean_price_ratio': 0.980378420107658, 'minimum_mean_from_floor': 0.0, 'target_attainable': True, 'multiplier': 1.154250413698586}
fleet_lease_bargaining {'target_mean_price_ratio': 0.9415885956004653, 'minimum_mean_from_floor': 0.0, 'target_attainable': True, 'multiplier': 1.0390858047803866}


Export schemas and metadata verified.
